In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LinearRegressionLab").getOrCreate()

In [3]:
data = spark.read.csv("california_housing.csv", header=True, inferSchema=True)
data.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+---------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|ocean_proximity|median_house_value|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+---------------+------------------+
|  -122.23|   37.88|                41|        880|           129|       322|       126|       8.3252|       NEAR BAY|            452600|
|  -122.22|   37.86|                21|       7099|          1106|      2401|      1138|       8.3014|       NEAR BAY|            358500|
|  -122.24|   37.85|                52|       1467|           190|       496|       177|       7.2574|       NEAR BAY|            352100|
|  -122.25|   37.85|                52|       1274|           235|       558|       219|       5.6431|       NEAR BAY|            341300|
|  -122.25|   37.85|              

In [4]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["median_income", "housing_median_age", "total_rooms", "population"],
    outputCol="features"
)

output = assembler.transform(data).select("features", "median_house_value")
output.show(5)

+--------------------+------------------+
|            features|median_house_value|
+--------------------+------------------+
|[8.3252,41.0,880....|            452600|
|[8.3014,21.0,7099...|            358500|
|[7.2574,52.0,1467...|            352100|
|[5.6431,52.0,1274...|            341300|
|[3.8462,52.0,1627...|            342200|
+--------------------+------------------+
only showing top 5 rows



In [5]:
train_data, test_data = output.randomSplit([0.7, 0.3])

In [6]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="median_house_value")
lr_model = lr.fit(train_data)

In [7]:
test_results = lr_model.evaluate(test_data)
print(f"RMSE: {test_results.rootMeanSquaredError}")
print(f"R²: {test_results.r2}")

RMSE: 80530.87177906994
R²: 0.5164007555606569
